# Q3 – LLM-based ESG message triage (BUS5001 Assessment 3)

Compares four approaches on 8 ESG operational messages with hand-labelled expected values:

1. **LLM, v1 prompt** (the prompt given in the brief)
2. **LLM, v2 prompt** (revised prompt)
3. **Zero-shot classifier** – `facebook/bart-large-mnli` (Hugging Face)
4. **Rule-based baseline** – keyword rules

LLM: `Qwen/Qwen2.5-1.5B-Instruct` (open-source, Apache-2.0), greedy decoding for reproducibility.
All messages are simulated; no real personal data is used.

**Run:** Runtime → Change runtime type → T4 GPU (optional, faster) → Runtime → Run all.

In [5]:
import sys, platform, json, re, time, datetime, os
import torch, transformers
print('python', sys.version.split()[0], '| torch', torch.__version__, '| transformers', transformers.__version__)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE=='cuda' else '')
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
os.makedirs('logs', exist_ok=True); os.makedirs('results', exist_ok=True); os.makedirs('prompts', exist_ok=True); os.makedirs('data', exist_ok=True)
print('run id:', RUN_ID)

python 3.13.15 | torch 2.11.0+cu130 | transformers 5.18.0
device: cuda Tesla T4
run id: 20261011T161818Z


## 1. Test messages and expected labels
M1–M5 are from the assessment brief; M6–M8 were written to test hazards, positive feedback and personal-data handling.

In [6]:
import pandas as pd
MESSAGES = [
 dict(id='M1', text='There is a water leak in Building C that has been running all morning.',
      exp_category='water_leak', exp_urgency='HIGH', exp_team='Facilities & Energy', exp_followup='Y', exp_sensitivity='LOW'),
 dict(id='M2', text='The recycling bins are contaminated again and no one seems to be checking them.',
      exp_category='waste_contamination', exp_urgency='MEDIUM', exp_team='Waste & Environment', exp_followup='Y', exp_sensitivity='LOW'),
 dict(id='M3', text='The air conditioning is running overnight in an empty office.',
      exp_category='energy_waste', exp_urgency='MEDIUM', exp_team='Facilities & Energy', exp_followup='Y', exp_sensitivity='LOW'),
 dict(id='M4', text='I want to report that one of our suppliers may not meet our sustainability policy.',
      exp_category='supplier_compliance', exp_urgency='MEDIUM', exp_team='Procurement', exp_followup='Y', exp_sensitivity='MEDIUM'),
 dict(id='M5', text='The accessible entrance near the main building has been blocked for two days.',
      exp_category='accessibility_barrier', exp_urgency='HIGH', exp_team='Facilities & Inclusion', exp_followup='Y', exp_sensitivity='LOW'),
 dict(id='M6', text="Someone dumped old paint tins and batteries behind Building D and there's a strong chemical smell near the staff entrance.",
      exp_category='hazardous_disposal', exp_urgency='CRITICAL', exp_team='Waste & Environment', exp_followup='Y', exp_sensitivity='LOW'),
 dict(id='M7', text='Great job on the new solar panels on the car park roof, the shade is lovely too!',
      exp_category='other', exp_urgency='LOW', exp_team='Sustainability Team', exp_followup='N', exp_sensitivity='LOW'),
 dict(id='M8', text="My manager, John Smith, told us to change the energy figures in this quarter's ESG report so they look better.",
      exp_category='governance_concern', exp_urgency='HIGH', exp_team='Governance & Ethics', exp_followup='Y', exp_sensitivity='HIGH'),
]
df = pd.DataFrame(MESSAGES)
df.to_csv('data/messages_labelled.csv', index=False)
CATEGORIES = ['water_leak','energy_waste','waste_contamination','hazardous_disposal','accessibility_barrier','supplier_compliance','governance_concern','other']
TEAM_FOR = {'water_leak':'Facilities & Energy','energy_waste':'Facilities & Energy','waste_contamination':'Waste & Environment',
            'hazardous_disposal':'Waste & Environment','accessibility_barrier':'Facilities & Inclusion','supplier_compliance':'Procurement',
            'governance_concern':'Governance & Ethics','other':'Sustainability Team'}
df[['id','text','exp_category','exp_urgency','exp_team']]

,id,text,exp_category,exp_urgency,exp_team
0,M1,There is a water leak in Building C that has b...,water_leak,HIGH,Facilities & Energy
1,M2,The recycling bins are contaminated again and ...,waste_contamination,MEDIUM,Waste & Environment
2,M3,The air conditioning is running overnight in a...,energy_waste,MEDIUM,Facilities & Energy
3,M4,I want to report that one of our suppliers may...,supplier_compliance,MEDIUM,Procurement
4,M5,The accessible entrance near the main building...,accessibility_barrier,HIGH,Facilities & Inclusion
5,M6,Someone dumped old paint tins and batteries be...,hazardous_disposal,CRITICAL,Waste & Environment
6,M7,Great job on the new solar panels on the car p...,other,LOW,Sustainability Team
7,M8,"My manager, John Smith, told us to change the ...",governance_concern,HIGH,Governance & Ethics


## 2. Prompts
**v1** is the brief's starting prompt. **v2** fixes its weaknesses: no allowed category list, no urgency rules, no team list, undefined sensitivity, no instruction for uncertainty or one-object output, and no schema.

In [7]:
PROMPT_V1 = """An employee has submitted an ESG-related operational message.
Extract the relevant issue details and return JSON only.
For each issue, identify:
- issue_category
- urgency: LOW, MEDIUM, HIGH or CRITICAL
- sentiment: POSITIVE, NEUTRAL or NEGATIVE
- followup_required: Y or N
- recommended_team
- escalation_reason
- data_sensitivity_risk
- brief_summary
Return JSON ONLY. No other text outside the JSON.
Message: {message}"""

PROMPT_V2 = """You are an ESG operations triage assistant for a large organisation. Classify ONE employee message.

Rules:
1. issue_category must be exactly one of: water_leak, energy_waste, waste_contamination, hazardous_disposal, accessibility_barrier, supplier_compliance, governance_concern, other.
2. urgency must be one of LOW, MEDIUM, HIGH, CRITICAL, using these definitions:
   - CRITICAL: immediate risk to people's safety or health (e.g. chemical smell, water near electrics, fire risk).
   - HIGH: ongoing resource loss or access barrier lasting hours or more, or alleged misconduct.
   - MEDIUM: problem that should be fixed within days, no immediate harm.
   - LOW: feedback, suggestions or questions.
3. recommended_team must be the team mapped to the category:
   water_leak or energy_waste -> "Facilities & Energy"; waste_contamination or hazardous_disposal -> "Waste & Environment";
   accessibility_barrier -> "Facilities & Inclusion"; supplier_compliance -> "Procurement";
   governance_concern -> "Governance & Ethics"; other -> "Sustainability Team".
4. sentiment: POSITIVE, NEUTRAL or NEGATIVE.
5. followup_required: "Y" if someone must act or reply, otherwise "N".
6. data_sensitivity_risk: "HIGH" if the message names or identifies a person, or contains health, conduct or legal allegations; "MEDIUM" if it names a company or supplier; otherwise "LOW".
7. escalation_reason: one short sentence on why it needs escalation, or "none".
8. needs_human_review: true if urgency is CRITICAL, data_sensitivity_risk is HIGH, category is governance_concern, or you are unsure.
9. brief_summary: max 20 words, using only facts stated in the message. Do not repeat personal names.
10. Never invent locations, names or facts. If unsure of the category, use "other".

Return ONE JSON object only, with exactly these keys:
{{"issue_category": "", "urgency": "", "sentiment": "", "followup_required": "", "recommended_team": "", "escalation_reason": "", "data_sensitivity_risk": "", "needs_human_review": false, "brief_summary": ""}}

Message: \"\"\"{message}\"\"\""""
open('prompts/v1_baseline_prompt.txt','w').write(PROMPT_V1)
open('prompts/v2_revised_prompt.txt','w').write(PROMPT_V2)
print(PROMPT_V2.format(message='<message text>'))

You are an ESG operations triage assistant for a large organisation. Classify ONE employee message.

Rules:
1. issue_category must be exactly one of: water_leak, energy_waste, waste_contamination, hazardous_disposal, accessibility_barrier, supplier_compliance, governance_concern, other.
2. urgency must be one of LOW, MEDIUM, HIGH, CRITICAL, using these definitions:
   - CRITICAL: immediate risk to people's safety or health (e.g. chemical smell, water near electrics, fire risk).
   - HIGH: ongoing resource loss or access barrier lasting hours or more, or alleged misconduct.
   - MEDIUM: problem that should be fixed within days, no immediate harm.
   - LOW: feedback, suggestions or questions.
3. recommended_team must be the team mapped to the category:
   water_leak or energy_waste -> "Facilities & Energy"; waste_contamination or hazardous_disposal -> "Waste & Environment";
   accessibility_barrier -> "Facilities & Inclusion"; supplier_compliance -> "Procurement";
   governance_concern -

## 3. Run the open-source LLM (Qwen2.5-1.5B-Instruct)

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
GEN = dict(max_new_tokens=320, do_sample=False)
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16 if DEVICE=='cuda' else torch.float32).to(DEVICE)
model.eval()

def extract_json(raw):
    m = re.search(r'\{.*\}', raw, re.S)
    if not m: return None, 'no JSON object found'
    try: return json.loads(m.group(0)), None
    except Exception as e: return None, f'invalid JSON: {e}'

def run_llm(prompt_template, text):
    msgs = [{'role':'user','content':prompt_template.format(message=text)}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt', return_dict=True).to(DEVICE); ids = enc['input_ids']
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**enc, **GEN, pad_token_id=tok.eos_token_id)
    raw = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
    parsed, err = extract_json(raw)
    return dict(raw=raw, parsed=parsed, parse_error=err, json_only=raw.startswith('{') and raw.endswith('}'), seconds=round(time.time()-t0,2))

LLM = {}
for version, tmpl in [('v1', PROMPT_V1), ('v2', PROMPT_V2)]:
    path = f'logs/{RUN_ID}_llm_{version}.jsonl'
    with open(path, 'w') as f:
        for m in MESSAGES:
            r = run_llm(tmpl, m['text'])
            rec = dict(run_id=RUN_ID, model=MODEL_ID, generation=GEN, prompt_version=version, message_id=m['id'], message=m['text'], **r)
            LLM[(version, m['id'])] = rec
            f.write(json.dumps(rec) + '\n')
            print(f"--- {version} {m['id']} ({r['seconds']}s) json_only={r['json_only']} error={r['parse_error']}\n{r['raw']}\n")
    print('saved', path)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

--- v1 M1 (6.6s) json_only=False error=None
```json
{
  "issue_category": "Infrastructure",
  "urgency": "HIGH",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "recommended_team": "Facilities Management Team",
  "escalation_reason": "Immediate action required to prevent potential damage.",
  "data_sensitivity_risk": "LOW",
  "brief_summary": "Water leak detected in Building C for several hours."
}
```

--- v1 M2 (4.02s) json_only=False error=None
```json
{
  "issue_category": "Operational",
  "urgency": "HIGH",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "recommended_team": "Operations Team",
  "escalation_reason": "Contamination of recycling bins is causing waste management issues.",
  "data_sensitivity_risk": "LOW",
  "brief_summary": "Recycling bins are contaminated again with no checks being made."
}
```

--- v1 M3 (3.67s) json_only=False error=None
```json
{
  "issue_category": "Environmental",
  "urgency": "HIGH",
  "sentiment": "NEGATIVE",
  "followup_requ

In [14]:
for k in ['M1','M6','M8']: print(k, json.dumps({'v1': LLM[('v1',k)]['raw'], 'v2': LLM[('v2',k)]['raw']}))

M1 {"v1": "```json\n{\n  \"issue_category\": \"Infrastructure\",\n  \"urgency\": \"HIGH\",\n  \"sentiment\": \"NEGATIVE\",\n  \"followup_required\": \"Y\",\n  \"recommended_team\": \"Facilities Management Team\",\n  \"escalation_reason\": \"Immediate action required to prevent potential damage.\",\n  \"data_sensitivity_risk\": \"LOW\",\n  \"brief_summary\": \"Water leak detected in Building C for several hours.\"\n}\n```", "v2": "```json\n{\n  \"issue_category\": \"water_leak\",\n  \"urgency\": \"HIGH\",\n  \"sentiment\": \"NEGATIVE\",\n  \"followup_required\": \"Y\",\n  \"recommended_team\": \"Facilities & Energy\",\n  \"escalation_reason\": \"Immediate risk to people's safety due to potential electrical hazards.\",\n  \"data_sensitivity_risk\": \"LOW\",\n  \"needs_human_review\": false,\n  \"brief_summary\": \"Water leak in Building C affecting electricity.\"\n}\n```"}
M6 {"v1": "```json\n{\n  \"issue_category\": \"Environmental\",\n  \"urgency\": \"HIGH\",\n  \"sentiment\": \"NEGATI

## 4. Baseline A – zero-shot classifier (facebook/bart-large-mnli)

In [9]:
from transformers import pipeline
zs = pipeline('zero-shot-classification', model='facebook/bart-large-mnli', device=0 if DEVICE=='cuda' else -1)
CAT_LABELS = {'water leak':'water_leak','energy waste':'energy_waste','recycling or waste contamination':'waste_contamination',
              'hazardous waste or chemical disposal':'hazardous_disposal','accessibility barrier':'accessibility_barrier',
              'supplier or procurement compliance':'supplier_compliance','misconduct or governance concern':'governance_concern',
              'positive feedback or general comment':'other'}
URG_LABELS = {'immediate danger to safety':'CRITICAL','urgent problem':'HIGH','routine problem':'MEDIUM','feedback or no action needed':'LOW'}
ZS = {}
with open(f'logs/{RUN_ID}_zero_shot.jsonl','w') as f:
    for m in MESSAGES:
        c = zs(m['text'], list(CAT_LABELS))
        u = zs(m['text'], list(URG_LABELS), hypothesis_template='This message describes a {}.')
        rec = dict(run_id=RUN_ID, model='facebook/bart-large-mnli', message_id=m['id'],
                   category=CAT_LABELS[c['labels'][0]], category_score=round(c['scores'][0],3),
                   urgency=URG_LABELS[u['labels'][0]], urgency_score=round(u['scores'][0],3),
                   all_category_scores=dict(zip([CAT_LABELS[l] for l in c['labels']],[round(s,3) for s in c['scores']])))
        rec['team'] = TEAM_FOR[rec['category']]
        ZS[m['id']] = rec; f.write(json.dumps(rec)+'\n')
pd.DataFrame(ZS.values())[['message_id','category','category_score','urgency','urgency_score','team']]

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,message_id,category,category_score,urgency,urgency_score,team
0,M1,water_leak,0.933,HIGH,0.717,Facilities & Energy
1,M2,waste_contamination,0.742,HIGH,0.428,Waste & Environment
2,M3,energy_waste,0.350,LOW,0.736,Facilities & Energy
3,M4,governance_concern,0.571,HIGH,0.626,Governance & Ethics
4,M5,accessibility_barrier,0.890,HIGH,0.678,Facilities & Inclusion
5,M6,hazardous_disposal,0.651,HIGH,0.640,Waste & Environment
6,M7,other,0.978,LOW,0.760,Sustainability Team
7,M8,other,0.580,HIGH,0.465,Sustainability Team


## 5. Baseline B – rule-based keyword classifier

In [10]:
RULES = [('hazardous_disposal', ['chemical','paint','batter','oil','hazard','asbestos','e-waste']),
         ('water_leak', ['leak','water','flood','pipe','tap','drip']),
         ('energy_waste', ['air conditioning','aircon','heating','lights','overnight','power','energy']),
         ('waste_contamination', ['recycl','bin','waste','rubbish','contaminat']),
         ('accessibility_barrier', ['accessib','ramp','lift','wheelchair','entrance blocked']),
         ('supplier_compliance', ['supplier','vendor','procure','contractor']),
         ('governance_concern', ['manager','fraud','falsif','bribe','misconduct','change the','report so'])]
def rule_classify(text):
    t = text.lower()
    cat = next((c for c, kws in RULES if any(k in t for k in kws)), 'other')
    if any(k in t for k in ['chemical','smell','injur','fire','electric','danger']): urg = 'CRITICAL'
    elif any(k in t for k in ['all morning','two days','days','again','manager']): urg = 'HIGH'
    elif any(k in t for k in ['great','thanks','lovely','suggest']): urg = 'LOW'
    else: urg = 'MEDIUM'
    return dict(category=cat, urgency=urg, team=TEAM_FOR[cat])
RB = {m['id']: dict(message_id=m['id'], **rule_classify(m['text'])) for m in MESSAGES}
with open(f'logs/{RUN_ID}_rule_based.jsonl','w') as f:
    for r in RB.values(): f.write(json.dumps(r)+'\n')
pd.DataFrame(RB.values())

,message_id,category,urgency,team
0,M1,water_leak,HIGH,Facilities & Energy
1,M2,waste_contamination,HIGH,Waste & Environment
2,M3,energy_waste,MEDIUM,Facilities & Energy
3,M4,supplier_compliance,MEDIUM,Procurement
4,M5,accessibility_barrier,HIGH,Facilities & Inclusion
5,M6,hazardous_disposal,CRITICAL,Waste & Environment
6,M7,other,LOW,Sustainability Team
7,M8,energy_waste,HIGH,Facilities & Energy


## 6. Comparison against expected labels

In [11]:
def g(rec, key):
    p = rec['parsed'] or {}
    return str(p.get(key, '—')).strip()
rows = []
for m in MESSAGES:
    v1, v2 = LLM[('v1', m['id'])], LLM[('v2', m['id'])]
    rows.append(dict(id=m['id'],
        expected_cat=m['exp_category'], v1_cat=g(v1,'issue_category'), v2_cat=g(v2,'issue_category'), zs_cat=ZS[m['id']]['category'], rule_cat=RB[m['id']]['category'],
        expected_urg=m['exp_urgency'], v1_urg=g(v1,'urgency'), v2_urg=g(v2,'urgency'), zs_urg=ZS[m['id']]['urgency'], rule_urg=RB[m['id']]['urgency'],
        expected_team=m['exp_team'], v1_team=g(v1,'recommended_team'), v2_team=g(v2,'recommended_team'),
        expected_sens=m['exp_sensitivity'], v1_sens=g(v1,'data_sensitivity_risk'), v2_sens=g(v2,'data_sensitivity_risk'), v2_review=g(v2,'needs_human_review')))
cmp = pd.DataFrame(rows)
cmp.to_csv(f'results/{RUN_ID}_comparison.csv', index=False)

def acc(col, exp):
    return round((cmp[col].str.lower() == cmp[exp].str.lower()).mean()*100)
summary = pd.DataFrame([
  dict(method='LLM v1 prompt', category_acc=acc('v1_cat','expected_cat'), urgency_acc=acc('v1_urg','expected_urg'), team_acc=acc('v1_team','expected_team'),
       valid_json=sum(LLM[('v1',m['id'])]['parsed'] is not None for m in MESSAGES), json_only=sum(LLM[('v1',m['id'])]['json_only'] for m in MESSAGES)),
  dict(method='LLM v2 prompt', category_acc=acc('v2_cat','expected_cat'), urgency_acc=acc('v2_urg','expected_urg'), team_acc=acc('v2_team','expected_team'),
       valid_json=sum(LLM[('v2',m['id'])]['parsed'] is not None for m in MESSAGES), json_only=sum(LLM[('v2',m['id'])]['json_only'] for m in MESSAGES)),
  dict(method='Zero-shot bart-large-mnli', category_acc=acc('zs_cat','expected_cat'), urgency_acc=acc('zs_urg','expected_urg'), team_acc=None, valid_json=None, json_only=None),
  dict(method='Rule-based keywords', category_acc=acc('rule_cat','expected_cat'), urgency_acc=acc('rule_urg','expected_urg'), team_acc=None, valid_json=None, json_only=None)])
summary.to_csv(f'results/{RUN_ID}_summary.csv', index=False)
print('Accuracy (%) on 8 messages; valid_json/json_only counts out of 8')
print(summary.to_markdown(index=False)); print(); print(cmp.to_markdown(index=False))

Accuracy (%) on 8 messages; valid_json/json_only counts out of 8
| method                    |   category_acc |   urgency_acc |   team_acc |   valid_json |   json_only |
|:--------------------------|---------------:|--------------:|-----------:|-------------:|------------:|
| LLM v1 prompt             |              0 |            50 |          0 |            8 |           0 |
| LLM v2 prompt             |             75 |            50 |         75 |            8 |           0 |
| Zero-shot bart-large-mnli |             75 |            50 |        nan |          nan |         nan |
| Rule-based keywords       |             88 |            88 |        nan |          nan |         nan |

| id   | expected_cat          | v1_cat              | v2_cat                | zs_cat                | rule_cat              | expected_urg   | v1_urg   | v2_urg   | zs_urg   | rule_urg   | expected_team          | v1_team                    | v2_team                | expected_sens   | v1_sens   | v2_se

## 7. Consistency check – rerun v2 three times on M1–M3 with sampling (temperature 0.7)

In [12]:
CONS = []
for m in MESSAGES[:3]:
    for i in range(3):
        msgs=[{'role':'user','content':PROMPT_V2.format(message=m['text'])}]
        enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt', return_dict=True).to(DEVICE); ids = enc['input_ids']
        with torch.no_grad(): out = model.generate(**enc, max_new_tokens=320, do_sample=True, temperature=0.7, top_p=0.9, pad_token_id=tok.eos_token_id)
        raw = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip(); p,_ = extract_json(raw); p = p or {}
        CONS.append(dict(message_id=m['id'], run=i+1, category=p.get('issue_category'), urgency=p.get('urgency'), team=p.get('recommended_team')))
cons = pd.DataFrame(CONS); cons.to_csv(f'results/{RUN_ID}_consistency.csv', index=False)
print(cons.to_markdown(index=False))

| message_id   |   run | category            | urgency   | team                |
|:-------------|------:|:--------------------|:----------|:--------------------|
| M1           |     1 | water_leak          | HIGH      | Facilities & Energy |
| M1           |     2 | water_leak          | HIGH      | Facilities & Energy |
| M1           |     3 | water_leak          | CRITICAL  | Facilities & Energy |
| M2           |     1 | waste_contamination | HIGH      | Waste & Environment |
| M2           |     2 | waste_contamination | HIGH      | Waste & Environment |
| M2           |     3 | waste_contamination | MEDIUM    | Waste & Environment |
| M3           |     1 | waste_contamination | LOW       | Waste & Environment |
| M3           |     2 | water_leak          | LOW       | Facilities & Energy |
| M3           |     3 | waste_contamination | LOW       | Waste & Environment |


## 8. Download all logs

In [13]:
import zipfile
zname = f'q3_outputs_{RUN_ID}.zip'
with zipfile.ZipFile(zname, 'w') as z:
    for d in ['logs','results','prompts','data']:
        for fn in os.listdir(d): z.write(os.path.join(d, fn))
        print(d, os.listdir(d))
try:
    from google.colab import files; files.download(zname)
except Exception as e: print('download manually:', e)

logs ['20261011T161818Z_zero_shot.jsonl', '20261011T161818Z_llm_v1.jsonl', '20261011T161818Z_llm_v2.jsonl', '20261011T161818Z_rule_based.jsonl', '20261011T161550Z_llm_v1.jsonl']
results ['20261011T161818Z_comparison.csv', '20261011T161818Z_summary.csv', '20261011T161818Z_consistency.csv']
prompts ['v2_revised_prompt.txt', 'v1_baseline_prompt.txt']
data ['messages_labelled.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>